# IVDC Lesson 8 · Colab 考試版（**完全自包含，不 clone**）

這個筆記本把 9 支腳本（8 支 demo + 1 支 benchmark）**整份內嵌**在下面的 `%%writefile` 格裡，
跑完那些格就會在你的 `/content` 產生 `src/` 與 `bench/`，之後用 `%run` 執行。

**唯一需要網路的地方**：下載模型權重與 MNIST（TF-Hub 約 20 MB、torchvision 權重、`yolov8n.pt` 6 MB）。
程式碼本身不需要 GitHub、不需要上傳 zip。

三條規則：
1. Colab **沒有 webcam、沒有螢幕** → 用 `--video clip.mp4`、不要 `imshow`。
2. **用 `%run`，不要 `!python`** —— `!python` 走 shell PATH，可能指到另一顆直譯器。
3. **`setuptools` 不用降版**（Colab 是 80.10.2，`pkg_resources` 還在）。

執行順序：往下依序跑。第 2 格是自檢，出問題就停在那裡。

## 1. 建目錄 + 環境自檢

In [ ]:
import os, sys, platform
os.makedirs("src", exist_ok=True)
os.makedirs("bench", exist_ok=True)
print(platform.platform(), "| Python", sys.version.split()[0])

import importlib
for m in ["numpy", "cv2", "torch", "torchvision", "tensorflow", "tensorflow_hub", "matplotlib", "ultralytics"]:
    try:
        mod = importlib.import_module(m)
        print(f"  OK      {m:16s} {getattr(mod, '__version__', '?')}")
    except Exception:
        print(f"  MISSING {m:16s}")

import torch
print("CUDA:", torch.cuda.is_available(), "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## 2. 補裝 ultralytics（只有 YOLO 那格需要）

In [ ]:
%pip install -q ultralytics
import ultralytics
print("ultralytics", ultralytics.__version__)

## 3. 測試素材：圖片 + 由圖片合成的影片（取代 webcam）

In [ ]:
import urllib.request, os, cv2

if not os.path.exists("bus.jpg"):
    urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg", "bus.jpg")
print("bus.jpg", os.path.getsize("bus.jpg"), "bytes")

img = cv2.imread("bus.jpg")
h, w = img.shape[:2]
writer = cv2.VideoWriter("clip.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 15.0, (w, h))
for _ in range(30):
    writer.write(img)
writer.release()
print("clip.mp4", os.path.getsize("clip.mp4"), "bytes", f"({w}x{h}, 30 frames)")

## 內嵌：`src/00_check_env.py`

In [ ]:
%%writefile src/00_check_env.py
#!/usr/bin/env python3
"""00_check_env.py — 相機 + 顯示環境自檢 (pre-exam environment check)。

IVDC Lesson 8「物件偵測與 CNN」講義還原腳本。每換一台機器（尤其是考試機）先跑這支。

用法:
    python 00_check_env.py
    python 00_check_env.py --strict          # 缺 GUI / 相機就回傳非 0
    python 00_check_env.py --skip-camera     # 考試機沒相機時

檢查四件事:
    1. 套件版本（缺哪個、怎麼裝）
    2. 有沒有 GUI backend —— 注意 hasattr(cv2, "imshow") 會騙人，
       一定要「真的呼叫一次」才算數
    3. 攝影機：不是 index 0 就是對的，要逐個開、量畫面亮度
    4. 前幾幀能不能用（自動曝光）

標註慣例
    # [原文]  簡報截圖上讀得到的程式碼
    # [補完]  截圖被滾動箭頭遮住／講義省略，依框架標準介面補完
"""

from __future__ import annotations

import argparse
import importlib
import platform
import sys

# [原文] 簡報自檢清單：缺哪個、就印出對應的 pip install
PACKAGES = [
    ("cv2", "opencv-python"),
    ("torch", "torch"),
    ("torchvision", "torchvision"),
    ("tensorflow", "tensorflow"),
    ("tensorflow_hub", "tensorflow-hub"),
    ("ultralytics", "ultralytics"),
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
]


def report_packages() -> dict[str, str]:
    """回傳 {module: version}，順便印出人看的表格。"""
    found: dict[str, str] = {}
    print("== 套件 ==")
    for module, package in PACKAGES:
        try:
            mod = importlib.import_module(module)
            version = getattr(mod, "__version__", "?")
            found[module] = version
            print(f"  OK      {module:16s} {version}")
        except Exception:
            print(f"  MISSING {module:16s} -> pip install {package}")
    print(f"  --      python           {platform.python_version()} ({sys.executable})")
    return found


def imshow_available() -> bool:
    """真的彈一次視窗才算數。"""
    # [原文] 這段是「坑四」的解法：不能只判斷函式存在
    import cv2
    import numpy as np

    try:
        cv2.imshow("__probe__", np.zeros((4, 4, 3), np.uint8))
        cv2.waitKey(1)
        cv2.destroyWindow("__probe__")
        return True
    except Exception:
        return False


def report_gui() -> bool:
    print("\n== 顯示 (GUI) ==")
    try:
        import cv2
    except Exception:
        print("  MISSING cv2 -> 無法檢查 GUI")
        return False

    # [原文] currentUIFramework() 回空字串 = 沒有 GUI backend
    try:
        framework = cv2.currentUIFramework()
    except Exception:
        framework = "(這個版本沒有 currentUIFramework())"
    print(f"  UI 框架: {framework or '(空 = 無)'}")

    ok = imshow_available()
    print("  OK      可以正常彈窗" if ok else "  失敗    imshow 不可用（headless 版 OpenCV）")
    if not ok:
        print("  修法:    pip uninstall opencv-python-headless -y && pip install opencv-python")
    return ok


def _backend() -> int:
    """Windows 上用 DirectShow；其他平台交給 OpenCV 自己挑。"""
    # [補完] 簡報只講 Windows（CAP_DSHOW）。原樣搬到 Linux 會每個 index 都開不起來，
    #        所以在這裡依平台選擇後端。
    import cv2

    return cv2.CAP_DSHOW if sys.platform == "win32" else cv2.CAP_ANY


def camera_backends() -> list[int]:
    """要依序嘗試的後端清單。"""
    import cv2

    # [原文] 坑二：Windows 上 CAP_MSMF 常常打不開，要依序重試
    if sys.platform == "win32":
        return [cv2.CAP_DSHOW, cv2.CAP_MSMF, cv2.CAP_ANY]
    return [cv2.CAP_ANY]


def pick_camera(preferred: int = 0, max_index: int = 6, min_brightness: float = 20.0):
    """回傳第一個「真的有畫面」的攝影機（已開好、已暖機）。

    [原文] 坑一：cv2.VideoCapture(0) 可能成功開啟卻是全黑，不會報錯。
    [補完] 暖機改成「時間」而不是固定幀數：自動曝光要 1～2 秒，
           只丟 10 幀（約 0.3 秒）根本不夠。
    """
    import cv2
    import time

    order = [preferred] + [i for i in range(max_index + 1) if i != preferred]
    for idx in order:
        for api in camera_backends():
            cap = cv2.VideoCapture(idx, api)
            if not cap.isOpened():
                cap.release()
                continue
            mean = -1.0
            t0 = time.time()
            while time.time() - t0 < 1.0:  # 等自動曝光穩定
                ok, frame = cap.read()
                if ok and frame is not None:
                    mean = float(frame.mean())
            if mean >= min_brightness:
                print(f"  OK      camera {idx} (backend {api}, brightness {mean:.0f})")
                return cap
            cap.release()
            print(f"           camera {idx} (backend {api}): black (mean {mean:.1f}) - skip")
    return None


def list_cameras(max_index: int = 6) -> bool:
    print("\n== 攝影機 ==")
    cap = pick_camera(0, max_index)
    if cap is None:
        print("  失敗    找不到有畫面的攝影機")
        print("  修法:    換 index、檢查裝置管理員，或直接跳過這項 (--skip-camera)")
        return False
    cap.release()
    return True


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description="IVDC Lesson 8 環境自檢")
    parser.add_argument("--skip-camera", action="store_true", help="不檢查攝影機")
    parser.add_argument("--max-camera-index", type=int, default=6, help="最多試到第幾個 index")
    parser.add_argument("--strict", action="store_true", help="GUI/相機不可用時回傳非 0")
    args = parser.parse_args(argv)

    print(f"== 平台 ==\n  {platform.platform()}\n")
    found = report_packages()
    gui_ok = report_gui()
    cam_ok = True if args.skip_camera else list_cameras(args.max_camera_index)

    print("\n== 結論 ==")
    model_paths = [name for name in ("torch", "tensorflow", "ultralytics") if name in found]
    print(f"  可用推論框架: {', '.join(model_paths) if model_paths else '（無）'}")
    print(f"  GUI: {'可以用 imshow' if gui_ok else '沒有，請走存檔模式'}")
    if args.skip_camera:
        cam_text = "未檢查（--skip-camera）"
    else:
        cam_text = "可用" if cam_ok else "不可用（請改用 --video 影片檔）"
    print(f"  相機: {cam_text}")

    if args.strict and not (gui_ok and cam_ok):
        return 1
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

## 內嵌：`src/01_tf_object_detection_image.py`

In [ ]:
%%writefile src/01_tf_object_detection_image.py
#!/usr/bin/env python3
"""01_tf_object_detection_image.py — TensorFlow/TF-Hub SSD MobileNet V2 單張圖偵測。

用法:
    python 01_tf_object_detection_image.py --image bus.jpg
    python 01_tf_object_detection_image.py --image bus.jpg --threshold 0.4 --show

模型網址與環境（實測 2026-11）
    簡報用的 https://tfhub.dev/tensorflow/ssd_mobilenet_v2/2 仍然可用：
    curl 會回 404（網頁層已下架），但 hub.load() 還是解析得到模型檔。
    真正會擋人的是 tensorflow_hub 依賴 pkg_resources，而 setuptools>=81 已經把它移除，
    於是 import tensorflow_hub 直接 ModuleNotFoundError。解法是釘住 setuptools：
        uv run --no-project --python 3.11 --with tensorflow-cpu \
            --with tensorflow-hub --with 'setuptools<81' -- python 01_...py
    想換模型（例如 Kaggle Models 上的同名版本）就用 --model-url。

標註慣例
    # [原文]  簡報截圖上讀得到的程式碼
    # [補完]  截圖被滾動箭頭遮住／講義省略，依框架標準介面補完

簡報對應：第 2 頁（Tensorflow for Object Detection），第 16 行以下被向下箭頭遮住。
"""

from __future__ import annotations

import argparse
import os
import sys

DEFAULT_MODEL_URL = "https://tfhub.dev/tensorflow/ssd_mobilenet_v2/2"

# COCO id -> 名稱（SSD MobileNet V2 的 labelmap；缺號代表該 id 在 COCO 裡沒用）
# [補完] 簡報只印 class id。要看得懂輸出就得有這張表。
COCO_LABELS = {
    1: "person", 2: "bicycle", 3: "car", 4: "motorcycle", 5: "airplane", 6: "bus",
    7: "train", 8: "truck", 9: "boat", 10: "traffic light", 11: "fire hydrant",
    13: "stop sign", 14: "parking meter", 15: "bench", 16: "bird", 17: "cat",
    18: "dog", 19: "horse", 20: "sheep", 21: "cow", 22: "elephant", 23: "bear",
    24: "zebra", 25: "giraffe", 27: "backpack", 28: "umbrella", 31: "handbag",
    32: "tie", 33: "suitcase", 34: "frisbee", 35: "skis", 36: "snowboard",
    37: "sports ball", 38: "kite", 39: "baseball bat", 40: "baseball glove",
    41: "skateboard", 42: "surfboard", 43: "tennis racket", 44: "bottle",
    46: "wine glass", 47: "cup", 48: "fork", 49: "knife", 50: "spoon", 51: "bowl",
    52: "banana", 53: "apple", 54: "sandwich", 55: "orange", 56: "broccoli",
    57: "carrot", 58: "hot dog", 59: "pizza", 60: "donut", 61: "cake", 62: "chair",
    63: "couch", 64: "potted plant", 65: "bed", 67: "dining table", 70: "toilet",
    72: "tv", 73: "laptop", 74: "mouse", 75: "remote", 76: "keyboard",
    77: "cell phone", 78: "microwave", 79: "oven", 80: "toaster", 81: "sink",
    82: "refrigerator", 84: "book", 85: "clock", 86: "vase", 87: "scissors",
    88: "teddy bear", 89: "hair drier", 90: "toothbrush",
}


def load_model(model_url: str):
    """載入 TF-Hub 模型；失敗時給出可執行的下一步，而不是丟一坨 traceback。"""
    import tensorflow_hub as hub

    try:
        return hub.load(model_url)
    except Exception as exc:
        print(f"[錯誤] 載入模型失敗: {model_url}", file=sys.stderr)
        print(f"        {type(exc).__name__}: {str(exc)[:300]}", file=sys.stderr)
        print("\n可能原因與解法：", file=sys.stderr)
        print("  1. setuptools>=81 移除了 pkg_resources → pip install 'setuptools<81'",
              file=sys.stderr)
        print("  2. 沒有網路 / 防火牆擋掉 tfhub.dev 與 storage.googleapis.com", file=sys.stderr)
        print("  3. 想換模型：--model-url <kaggle 或 tfhub 網址>", file=sys.stderr)
        print("  4. 先確認環境：python 00_check_env.py", file=sys.stderr)
        return None


def imshow_available() -> bool:
    # [原文] 坑四：不能只判斷函式存在，要真的呼叫一次
    import cv2
    import numpy as np

    try:
        cv2.imshow("__probe__", np.zeros((4, 4, 3), np.uint8))
        cv2.waitKey(1)
        cv2.destroyWindow("__probe__")
        return True
    except Exception:
        return False


def draw(frame, boxes, scores, classes, threshold: float):
    """在畫面上畫框 + 標籤。boxes 是歸一化 (ymin, xmin, ymax, xmax)。"""
    import cv2

    h, w = frame.shape[:2]
    for box, score, cls in zip(boxes, scores, classes):
        if score < threshold:
            continue
        # [原文] 座標是 0～1 的歸一化值，順序是 ymin, xmin, ymax, xmax
        ymin, xmin, ymax, xmax = box
        p0 = (int(xmin * w), int(ymin * h))
        p1 = (int(xmax * w), int(ymax * h))
        cv2.rectangle(frame, p0, p1, (0, 0, 255), 2)
        label = f"{COCO_LABELS.get(int(cls), 'class_' + str(int(cls)))} {score:.2f}"
        cv2.putText(frame, label, (p0[0], max(p0[1] - 6, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
    return frame


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description="TF-Hub SSD MobileNet V2 單張圖偵測")
    parser.add_argument("--image", required=True, help="輸入圖片路徑")
    parser.add_argument("--model-url",
                        default=os.environ.get("IVDC_TF_MODEL_URL", DEFAULT_MODEL_URL))
    parser.add_argument("--threshold", type=float, default=0.5, help="信心門檻（預設 0.5）")
    parser.add_argument("--out", default="out/tf_detection.jpg", help="輸出標註圖")
    parser.add_argument("--show", action="store_true", help="彈窗顯示（需要 GUI）")
    args = parser.parse_args(argv)

    import cv2
    import tensorflow as tf

    print(f"Python {sys.version.split()[0]} / TF {tf.__version__}")
    print(f"模型: {args.model_url}")
    model = load_model(args.model_url)
    if model is None:
        return 2

    # [原文] 讀圖 → BGR 轉 RGB → uint8 張量 → 加 batch 維度
    image = cv2.imread(args.image)
    if image is None:
        print(f"[錯誤] 讀不到圖片: {args.image}", file=sys.stderr)
        return 2
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    input_tensor = tf.convert_to_tensor(image_rgb, dtype=tf.uint8)
    input_tensor = tf.expand_dims(input_tensor, 0)  # (1, H, W, 3)

    # [原文] model(input_tensor) 一次做完前處理 + 推論 + NMS
    detections = model(input_tensor)

    # [補完] 簡報截圖到此為止（被右下角滾動箭頭蓋住），以下依 TF-Hub 標準介面補完
    boxes = detections["detection_boxes"][0].numpy()
    scores = detections["detection_scores"][0].numpy()
    classes = detections["detection_classes"][0].numpy().astype(int)

    kept = [(b, s, c) for b, s, c in zip(boxes, scores, classes) if s >= args.threshold]
    print(f"\n檢出 {len(kept)} 個（門檻 {args.threshold}）")
    for box, score, cls in kept:
        name = COCO_LABELS.get(int(cls), "class_" + str(int(cls)))
        ymin, xmin, ymax, xmax = box
        print(f"  {name:16s} score={score:.3f} "
              f"box=({xmin:.3f}, {ymin:.3f}, {xmax:.3f}, {ymax:.3f}) 歸一化")

    annotated = draw(image.copy(), boxes, scores, classes, args.threshold)
    out_dir = os.path.dirname(os.path.abspath(args.out))
    os.makedirs(out_dir, exist_ok=True)
    cv2.imwrite(args.out, annotated)
    print(f"\n已存檔: {args.out}")

    if args.show:
        if not imshow_available():
            print("[提醒] 這台機器沒有 GUI backend，只輸出檔案（見 00_check_env.py）")
            return 0
        cv2.imshow("TF SSD MobileNet V2", annotated)
        cv2.waitKey(0)
        cv2.destroyAllWindows()
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

## 內嵌：`src/02_tf_webcam_detection.py`

In [ ]:
%%writefile src/02_tf_webcam_detection.py
#!/usr/bin/env python3
"""02_tf_webcam_detection.py — TensorFlow/TF-Hub SSD MobileNet V2 即時 webcam 偵測。

用法:
    python 02_tf_webcam_detection.py
    python 02_tf_webcam_detection.py --camera 2 --threshold 0.4
    python 02_tf_webcam_detection.py --no-show --max-frames 60   # 無 GUI 機器煙霧測試

四個坑都解掉了（對應筆記 §十）：
    坑一 相機不一定在 index 0      -> pick_camera() 逐個開、量亮度
    坑二 Windows 要 DirectShow      -> 依序試 DSHOW / MSMF / ANY
    坑三 開機前幾幀不能用           -> 依「時間」暖機 1.5 秒（不是固定 10 幀）
    坑四 imshow 沒有 fallback 會崩  -> imshow_available()，沒有 GUI 就存檔

標註慣例
    # [原文]  簡報截圖上讀得到的程式碼
    # [補完]  截圖被滾動箭頭遮住／講義省略，依框架標準介面補完

簡報對應：第 3 頁（Real Time Webcam Detection），第 16 行以下被向下箭頭遮住。
"""

from __future__ import annotations

import argparse
import os
import sys
import time

DEFAULT_MODEL_URL = "https://tfhub.dev/tensorflow/ssd_mobilenet_v2/2"


def _load_labels() -> dict:
    """標籤表沿用 01（避免兩份 COCO 對照表各自走鐘）；找不到就退回 class id。"""
    # [補完] 簡報的 webcam 版沒有印類別名稱
    try:
        sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
        from importlib import import_module

        return import_module("01_tf_object_detection_image").COCO_LABELS
    except Exception:
        return {}


def imshow_available() -> bool:
    # [原文] 坑四
    import cv2
    import numpy as np

    try:
        cv2.imshow("__probe__", np.zeros((4, 4, 3), np.uint8))
        cv2.waitKey(1)
        cv2.destroyWindow("__probe__")
        return True
    except Exception:
        return False


def camera_backends() -> list[int]:
    # [原文] 坑二：Windows 上 CAP_MSMF 常常打不開
    import cv2

    if sys.platform == "win32":
        return [cv2.CAP_DSHOW, cv2.CAP_MSMF, cv2.CAP_ANY]
    return [cv2.CAP_ANY]


def pick_camera(preferred: int = 0, max_index: int = 6, min_brightness: float = 20.0):
    # [原文] 坑一：VideoCapture(0) 可能開得起來但全黑，不會報錯
    import cv2

    order = [preferred] + [i for i in range(max_index + 1) if i != preferred]
    for idx in order:
        for api in camera_backends():
            cap = cv2.VideoCapture(idx, api)
            if not cap.isOpened():
                cap.release()
                continue
            mean = -1.0
            t0 = time.time()
            while time.time() - t0 < 1.0:
                ok, frame = cap.read()
                if ok and frame is not None:
                    mean = float(frame.mean())
            if mean >= min_brightness:
                print(f"using camera {idx} (backend {api}, brightness {mean:.0f})")
                return cap
            cap.release()
            print(f"camera {idx} (backend {api}): black (mean {mean:.1f}) - skip")
    return None


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description="TF-Hub SSD MobileNet V2 即時偵測")
    parser.add_argument("--camera", type=int, default=0, help="偏好的攝影機 index")
    parser.add_argument("--video", default="",
                        help="用影片檔當來源（Colab／伺服器沒有 webcam 時用這個）")
    parser.add_argument("--model-url",
                        default=os.environ.get("IVDC_TF_MODEL_URL", DEFAULT_MODEL_URL))
    parser.add_argument("--threshold", type=float, default=0.5)
    parser.add_argument("--warmup", type=float, default=1.5, help="暖機秒數（自動曝光）")
    parser.add_argument("--max-frames", type=int, default=0, help="跑幾個 frame 就停（0=不限）")
    parser.add_argument("--no-show", action="store_true", help="不要彈窗，改成存圖")
    parser.add_argument("--out-dir", default="out/webcam_tf", help="--no-show 時的存圖目錄")
    args = parser.parse_args(argv)

    import cv2
    import tensorflow as tf
    import tensorflow_hub as hub

    labels = _load_labels()
    print(f"Python {sys.version.split()[0]} / TF {tf.__version__}")
    try:
        model = hub.load(args.model_url)
    except Exception as exc:
        print(f"[錯誤] 載入模型失敗: {args.model_url}", file=sys.stderr)
        print(f"        {type(exc).__name__}: {str(exc)[:200]}", file=sys.stderr)
        print("        常見原因：setuptools>=81 移除 pkg_resources（pip install 'setuptools<81'）",
              file=sys.stderr)
        return 2

    # [補完] 沒有 webcam 的環境（Colab／伺服器）：--video 用影片檔取代攝影機
    if args.video:
        cap = cv2.VideoCapture(args.video)
        if not cap.isOpened():
            print(f"[錯誤] 開不起影片: {args.video}", file=sys.stderr)
            return 2
        print(f"來源: 影片 {args.video}")
        args.warmup = 0.0
    else:
        cap = pick_camera(args.camera)
        if cap is None:
            print("[錯誤] 找不到有畫面的攝影機（先跑 00_check_env.py；"
                  "沒有 webcam 請用 --video 影片檔）", file=sys.stderr)
            return 2

    show_gui = (not args.no_show) and imshow_available()
    if not show_gui:
        os.makedirs(args.out_dir, exist_ok=True)
        print(f"[提醒] 沒有 GUI backend，改成存圖到 {args.out_dir}/")

    # [原文] 坑三：前幾幀不能用。簡報用固定 10 幀，這裡用時間（10 幀≈0.3 秒，不夠）
    t0 = time.time()
    while time.time() - t0 < args.warmup:
        cap.read()

    frames = 0
    fps = 0.0
    last = time.time()
    saved = 0
    processed = 0
    t_start = time.time()
    try:
        while True:
            ret, frame = cap.read()
            if not ret or frame is None:
                if args.video:
                    print("影片播完")
                else:
                    print("[錯誤] 讀不到 frame（相機被拔掉？）", file=sys.stderr)
                break
            processed += 1

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            # [原文] 同一套前處理：uint8 張量 + batch 維度
            input_tensor = tf.convert_to_tensor(rgb, dtype=tf.uint8)
            input_tensor = tf.expand_dims(input_tensor, 0)

            detections = model(input_tensor)

            # [補完] 門檻過濾 + 畫框（簡報截圖沒這段）
            h, w = frame.shape[:2]
            boxes = detections["detection_boxes"][0].numpy()
            scores = detections["detection_scores"][0].numpy()
            classes = detections["detection_classes"][0].numpy().astype(int)
            for box, score, cls in zip(boxes, scores, classes):
                if score < args.threshold:
                    continue
                ymin, xmin, ymax, xmax = box
                p0 = (int(xmin * w), int(ymin * h))
                p1 = (int(xmax * w), int(ymax * h))
                cv2.rectangle(frame, p0, p1, (0, 0, 255), 2)
                label = f"{labels.get(int(cls), 'class_' + str(int(cls)))} {score:.2f}"
                cv2.putText(frame, label, (p0[0], max(p0[1] - 6, 12)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)

            frames += 1
            now = time.time()
            if now - last >= 0.5:
                fps = frames / (now - last)
                frames, last = 0, now
            cv2.putText(frame, f"{fps:5.1f} FPS", (10, 28),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2, cv2.LINE_AA)

            if show_gui:
                cv2.imshow("TF Detection", frame)
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break
            else:
                path = os.path.join(args.out_dir, f"frame_{saved:04d}.jpg")
                cv2.imwrite(path, frame)
                saved += 1

            if args.max_frames and processed >= args.max_frames:
                break
    finally:
        cap.release()
        if show_gui:
            cv2.destroyAllWindows()

    if not show_gui:
        print(f"已存 {saved} 張到 {args.out_dir}/")
    if fps <= 0 and processed:
        fps = processed / max(time.time() - t_start, 1e-6)
    print(f"平均速度: {fps:.1f} FPS（{processed} 幀）")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

## 內嵌：`src/03_torch_object_detection_image.py`

In [ ]:
%%writefile src/03_torch_object_detection_image.py
#!/usr/bin/env python3
"""03_torch_object_detection_image.py — torchvision Faster R-CNN 單張圖偵測。

用法:
    python 03_torch_object_detection_image.py --image bus.jpg
    python 03_torch_object_detection_image.py --image bus.jpg --model fasterrcnn_mobilenet_320
    python 03_torch_object_detection_image.py --image bus.jpg --min-size 400   # CPU 加速關鍵

模型名稱（重要）
    筆記 §六 的表格用了 `fasterrcnn_resnet50` / `fasterrcnn_mobilenet` 這種簡寫，
    但 torchvision 裡真正的函式名只有：
        fasterrcnn_resnet50_fpn, fasterrcnn_resnet50_fpn_v2,
        fasterrcnn_mobilenet_v3_large_fpn, fasterrcnn_mobilenet_v3_large_320_fpn,
        ssdlite320_mobilenet_v3_large, retinanet_resnet50_fpn(_v2),
        fcos_resnet50_fpn, ssd300_vgg16
    直接 getattr 簡寫會 AttributeError，所以這裡有一張別名表。

標註慣例
    # [原文]  簡報截圖上讀得到的程式碼
    # [補完]  截圖被滾動箭頭遮住／講義省略，依框架標準介面補完

簡報對應：第 5 頁（Pytorch for Object Detection），第 12 行以下被向下箭頭遮住。
"""

from __future__ import annotations

import argparse
import os
import sys

# [補完] 簡寫 → torchvision 真實函式名（筆記表格的簡寫照抄會炸，這裡補起來）
MODEL_ALIASES = {
    "fasterrcnn_resnet50": "fasterrcnn_resnet50_fpn",
    "fasterrcnn_resnet50_v2": "fasterrcnn_resnet50_fpn_v2",
    "fasterrcnn_mobilenet": "fasterrcnn_mobilenet_v3_large_fpn",
    "fasterrcnn_mobilenet_320": "fasterrcnn_mobilenet_v3_large_320_fpn",
    "ssdlite320": "ssdlite320_mobilenet_v3_large",
    "retinanet": "retinanet_resnet50_fpn_v2",
}

# [補完] SSD/SSDLite 的 model 物件沒有 .weights 屬性（實測 AttributeError），
#        要靠 WeightsEnum 才拿得到類別名稱，所以備一張 模型 → WeightsEnum 的表。
WEIGHTS_ENUM = {
    "fasterrcnn_resnet50_fpn": "FasterRCNN_ResNet50_FPN_Weights",
    "fasterrcnn_resnet50_fpn_v2": "FasterRCNN_ResNet50_FPN_V2_Weights",
    "fasterrcnn_mobilenet_v3_large_fpn": "FasterRCNN_MobileNet_V3_Large_FPN_Weights",
    "fasterrcnn_mobilenet_v3_large_320_fpn": "FasterRCNN_MobileNet_V3_Large_320_FPN_Weights",
    "ssdlite320_mobilenet_v3_large": "SSDLite320_MobileNet_V3_Large_Weights",
    "ssd300_vgg16": "SSD300_VGG16_Weights",
    "retinanet_resnet50_fpn": "RetinaNet_ResNet50_FPN_Weights",
    "retinanet_resnet50_fpn_v2": "RetinaNet_ResNet50_FPN_V2_Weights",
    "fcos_resnet50_fpn": "FCOS_ResNet50_FPN_Weights",
}


def categories_for(resolved: str, model) -> list[str]:
    """COCO 類別名稱清單（取不到就回空 list，呼叫端退回印 class id）。"""
    weights = getattr(model, "weights", None)
    if weights is not None:
        return list(weights.meta["categories"])
    enum = WEIGHTS_ENUM.get(resolved)
    if enum:
        try:
            from torchvision.models import get_weight

            return list(get_weight(f"{enum}.DEFAULT").meta["categories"])
        except Exception:
            pass
    return []


def build_model(name: str, min_size: int | None, max_size: int | None):
    """依名稱建模型；回傳 (model, 解析後的函式名, 類別名稱清單)。"""
    import torchvision
    from torchvision.models import detection

    resolved = MODEL_ALIASES.get(name, name)
    if not hasattr(detection, resolved):
        available = sorted(n for n in dir(detection)
                           if n[0].islower() and not n.startswith("_"))
        print(f"[錯誤] torchvision 沒有 {name!r}（解析為 {resolved!r}）", file=sys.stderr)
        print("        可用的模型：", ", ".join(available), file=sys.stderr)
        return None, resolved, []

    # [原文] 這一行是簡報的重點
    factory = getattr(detection, resolved)
    kwargs = {"weights": "DEFAULT"}
    # [補完] CPU 加速：預設 min_size=800 會把 720p 短邊放大到 800，這步吃掉大半時間
    if min_size is not None:
        kwargs["min_size"] = min_size
    if max_size is not None:
        kwargs["max_size"] = max_size
    model = factory(**kwargs)
    # [原文] 一定要呼叫：train 模式回傳的是 loss，不是預測框
    model.eval()
    return model, resolved, categories_for(resolved, model)


def imshow_available() -> bool:
    # [原文] 坑四
    import cv2
    import numpy as np

    try:
        cv2.imshow("__probe__", np.zeros((4, 4, 3), np.uint8))
        cv2.waitKey(1)
        cv2.destroyWindow("__probe__")
        return True
    except Exception:
        return False


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description="torchvision 偵測模型單張圖")
    parser.add_argument("--image", required=True)
    parser.add_argument("--model", default="fasterrcnn_resnet50_fpn",
                        help="模型名或別名（預設簡報用的 fasterrcnn_resnet50_fpn）")
    parser.add_argument("--threshold", type=float, default=0.5)
    parser.add_argument("--min-size", type=int, default=None,
                        help="覆寫 min_size（CPU 上調小可大幅加速，例如 400）")
    parser.add_argument("--max-size", type=int, default=None)
    parser.add_argument("--out", default="out/torch_detection.jpg")
    parser.add_argument("--show", action="store_true")
    args = parser.parse_args(argv)

    import cv2
    import torch
    import torchvision
    from PIL import Image
    from torchvision import transforms

    print(f"PyTorch {torch.__version__} / torchvision {torchvision.__version__}")
    model, resolved, categories = build_model(args.model, args.min_size, args.max_size)
    if model is None:
        return 2
    print(f"模型: {resolved}")

    image = Image.open(args.image).convert("RGB")
    # [原文] ToTensor() 會轉成 (3,H,W) 且縮放到 0～1，正是 torchvision 偵測模型要的範圍
    transform = transforms.ToTensor()
    tensor = transform(image).unsqueeze(0)  # (1, 3, H, W)

    # [補完] 簡報截圖到此為止，以下依 torchvision 標準介面補完
    with torch.no_grad():
        prediction = model(tensor)[0]

    boxes = prediction["boxes"].numpy()    # 絕對像素座標，不是歸一化
    scores = prediction["scores"].numpy()  # 已依分數降序
    labels = prediction["labels"].numpy()

    frame = cv2.cvtColor(cv2.imread(args.image), cv2.COLOR_BGR2RGB)
    kept = 0
    print(f"\n檢出（門檻 {args.threshold}）:")
    for box, score, label in zip(boxes, scores, labels):
        if score < args.threshold:
            break  # [原文] 已依分數降序，可以直接 break
        x0, y0, x1, y1 = [int(v) for v in box]
        name = categories[int(label)] if 0 <= int(label) < len(categories) else f"class_{int(label)}"
        print(f"  {name:16s} score={score:.3f} box=({x0}, {y0}, {x1}, {y1}) 絕對像素")
        cv2.rectangle(frame, (x0, y0), (x1, y1), (0, 255, 0), 2)
        cv2.putText(frame, f"{name} {score:.2f}", (x0, max(y0 - 6, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)
        kept += 1
    print(f"共 {kept} 個")

    annotated = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
    os.makedirs(os.path.dirname(os.path.abspath(args.out)), exist_ok=True)
    cv2.imwrite(args.out, annotated)
    print(f"已存檔: {args.out}")

    if args.show:
        if not imshow_available():
            print("[提醒] 沒有 GUI backend，只輸出檔案")
            return 0
        cv2.imshow("PyTorch Detection", annotated)
        cv2.waitKey(0)
        cv2.destroyAllWindows()
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

## 內嵌：`src/04_torch_webcam_detection.py`

In [ ]:
%%writefile src/04_torch_webcam_detection.py
#!/usr/bin/env python3
"""04_torch_webcam_detection.py — torchvision 偵測模型即時 webcam（多模型可切換）。

用法:
    python 04_torch_webcam_detection.py
    python 04_torch_webcam_detection.py --model fasterrcnn_mobilenet_320
    python 04_torch_webcam_detection.py --model fasterrcnn_resnet50_fpn --min-size 400
    python 04_torch_webcam_detection.py --no-show --max-frames 30    # 無 GUI 煙霧測試

速度與模型別名
    `--model` 接受 torchvision 真名或筆記裡的簡寫（見 03 的 MODEL_ALIASES）。
    CPU 上真正該調的是 `--min-size`：預設 800 會把 720p 短邊放大到 800，
    調到 400 通常可拿回 2～3 倍 FPS。

標註慣例
    # [原文]  簡報截圖上讀得到的程式碼
    # [補完]  截圖被滾動箭頭遮住／講義省略，依框架標準介面補完

簡報對應：第 6 頁（Real Time Webcam Detection），第 13 行以下被向下箭頭遮住。
"""

from __future__ import annotations

import argparse
import os
import sys
import time

# [補完] 簡寫 → torchvision 真實函式名（與 03 相同，避免兩份表走鐘就從 03 借）
MODEL_ALIASES = {
    "fasterrcnn_resnet50": "fasterrcnn_resnet50_fpn",
    "fasterrcnn_resnet50_v2": "fasterrcnn_resnet50_fpn_v2",
    "fasterrcnn_mobilenet": "fasterrcnn_mobilenet_v3_large_fpn",
    "fasterrcnn_mobilenet_320": "fasterrcnn_mobilenet_v3_large_320_fpn",
    "ssdlite320": "ssdlite320_mobilenet_v3_large",
    "retinanet": "retinanet_resnet50_fpn_v2",
}


def categories_for(resolved: str, model) -> list[str]:
    """COCO 類別名稱。SSD 家族的 model 沒有 .weights 屬性，退回 WeightsEnum。"""
    # [補完] 實測 ssdlite320 沒有 .weights，直接取會 AttributeError
    weights = getattr(model, "weights", None)
    if weights is not None:
        return list(weights.meta["categories"])
    enum = {
        "ssdlite320_mobilenet_v3_large": "SSDLite320_MobileNet_V3_Large_Weights",
        "ssd300_vgg16": "SSD300_VGG16_Weights",
    }.get(resolved)
    if enum:
        try:
            from torchvision.models import get_weight

            return list(get_weight(f"{enum}.DEFAULT").meta["categories"])
        except Exception:
            pass
    return []


def imshow_available() -> bool:
    # [原文] 坑四
    import cv2
    import numpy as np

    try:
        cv2.imshow("__probe__", np.zeros((4, 4, 3), np.uint8))
        cv2.waitKey(1)
        cv2.destroyWindow("__probe__")
        return True
    except Exception:
        return False


def camera_backends() -> list[int]:
    # [原文] 坑二
    import cv2

    if sys.platform == "win32":
        return [cv2.CAP_DSHOW, cv2.CAP_MSMF, cv2.CAP_ANY]
    return [cv2.CAP_ANY]


def pick_camera(preferred: int = 0, max_index: int = 6, min_brightness: float = 20.0):
    # [原文] 坑一
    import cv2

    order = [preferred] + [i for i in range(max_index + 1) if i != preferred]
    for idx in order:
        for api in camera_backends():
            cap = cv2.VideoCapture(idx, api)
            if not cap.isOpened():
                cap.release()
                continue
            mean = -1.0
            t0 = time.time()
            while time.time() - t0 < 1.0:
                ok, frame = cap.read()
                if ok and frame is not None:
                    mean = float(frame.mean())
            if mean >= min_brightness:
                print(f"using camera {idx} (backend {api}, brightness {mean:.0f})")
                return cap
            cap.release()
            print(f"camera {idx} (backend {api}): black (mean {mean:.1f}) - skip")
    return None


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description="torchvision 偵測模型即時 webcam")
    parser.add_argument("--camera", type=int, default=0)
    parser.add_argument("--video", default="",
                        help="用影片檔當來源（Colab／伺服器沒有 webcam 時用這個）")
    parser.add_argument("--model", default="fasterrcnn_resnet50_fpn")
    parser.add_argument("--threshold", type=float, default=0.5)
    parser.add_argument("--min-size", type=int, default=None,
                        help="覆寫 min_size（CPU 上調小可大幅加速，例如 400）")
    parser.add_argument("--max-size", type=int, default=None)
    parser.add_argument("--warmup", type=float, default=1.5)
    parser.add_argument("--max-frames", type=int, default=0)
    parser.add_argument("--no-show", action="store_true")
    parser.add_argument("--out-dir", default="out/webcam_torch")
    args = parser.parse_args(argv)

    import cv2
    import torch
    import torchvision
    from PIL import Image
    from torchvision import transforms
    from torchvision.models import detection

    resolved = MODEL_ALIASES.get(args.model, args.model)
    if not hasattr(detection, resolved):
        available = sorted(n for n in dir(detection) if n[0].islower() and not n.startswith("_"))
        print(f"[錯誤] torchvision 沒有 {args.model!r}（解析為 {resolved!r}）", file=sys.stderr)
        print("        可用的模型：", ", ".join(available), file=sys.stderr)
        return 2

    kwargs = {"weights": "DEFAULT"}
    if args.min_size is not None:
        kwargs["min_size"] = args.min_size
    if args.max_size is not None:
        kwargs["max_size"] = args.max_size
    print(f"PyTorch {torch.__version__} / torchvision {torchvision.__version__}")
    print(f"模型: {resolved}" + (f" (min_size={args.min_size})" if args.min_size else ""))
    model = getattr(detection, resolved)(**kwargs)
    model.eval()
    categories = categories_for(resolved, model)
    transform = transforms.ToTensor()

    # [補完] Colab／伺服器沒有 webcam：--video 用影片檔取代攝影機，其餘流程完全一樣
    if args.video:
        cap = cv2.VideoCapture(args.video)
        if not cap.isOpened():
            print(f"[錯誤] 開不起影片: {args.video}", file=sys.stderr)
            return 2
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        print(f"來源: 影片 {args.video}（{n} 幀）")
        args.warmup = 0.0
    else:
        cap = pick_camera(args.camera)
        if cap is None:
            print("[錯誤] 找不到有畫面的攝影機（先跑 00_check_env.py；"
                  "沒有 webcam 的環境請用 --video 影片檔）", file=sys.stderr)
            return 2

    show_gui = (not args.no_show) and imshow_available()
    if not show_gui:
        os.makedirs(args.out_dir, exist_ok=True)
        print(f"[提醒] 沒有 GUI backend，改成存圖到 {args.out_dir}/")

    # [原文] 坑三：暖機（用時間，不用固定幀數）
    t0 = time.time()
    while time.time() - t0 < args.warmup:
        cap.read()

    processed = 0
    saved = 0
    fps = 0.0
    count = 0
    last = time.time()
    t_start = time.time()
    try:
        while True:
            ret, frame = cap.read()
            if not ret or frame is None:
                if args.video:
                    print("影片播完")          # 影片 EOF 不是錯誤
                else:
                    print("[錯誤] 讀不到 frame", file=sys.stderr)
                break
            processed += 1

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            # [原文] PIL → ToTensor → 加 batch 維度
            tensor = transform(Image.fromarray(rgb)).unsqueeze(0)

            with torch.no_grad():
                prediction = model(tensor)[0]

            # [原文] 已依分數降序，可以直接 break
            for box, score, label in zip(prediction["boxes"].numpy(),
                                         prediction["scores"].numpy(),
                                         prediction["labels"].numpy()):
                if score < args.threshold:
                    break
                x0, y0, x1, y1 = [int(v) for v in box]
                cv2.rectangle(frame, (x0, y0), (x1, y1), (0, 255, 0), 2)
                name = (categories[int(label)] if 0 <= int(label) < len(categories)
                        else f"class_{int(label)}")
                cv2.putText(frame, f"{name} {score:.2f}", (x0, max(y0 - 6, 12)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)

            count += 1
            now = time.time()
            if now - last >= 0.5:
                fps = count / (now - last)
                count, last = 0, now
            cv2.putText(frame, f"{fps:5.1f} FPS", (10, 28),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2, cv2.LINE_AA)

            if show_gui:
                cv2.imshow("PyTorch Detection", frame)
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break
            else:
                cv2.imwrite(os.path.join(args.out_dir, f"frame_{saved:04d}.jpg"), frame)
                saved += 1

            if args.max_frames and processed >= args.max_frames:
                break
    finally:
        cap.release()
        if show_gui:
            cv2.destroyAllWindows()

    if not show_gui:
        print(f"已存 {saved} 張到 {args.out_dir}/")
    # 影片很短或 --max-frames 很小時，0.5 秒的取樣窗可能還沒關過，退回平均值
    if fps <= 0 and processed:
        fps = processed / max(time.time() - t_start, 1e-6)
    print(f"平均速度: {fps:.1f} FPS（{processed} 幀）")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

## 內嵌：`src/05_yolo_detection.py`

In [ ]:
%%writefile src/05_yolo_detection.py
#!/usr/bin/env python3
"""05_yolo_detection.py — YOLOv8 偵測（簡報標「Recommended」的四行方案）。

用法:
    python 05_yolo_detection.py --source bus.jpg            # 單張圖
    python 05_yolo_detection.py --source 0 --show           # webcam 即時
    python 05_yolo_detection.py --source 0 --no-show --max-frames 30
    python 05_yolo_detection.py --source bus.jpg --imgsz 1280   # 小目標更完整，但更慢

為什麼預設是 yolov8n
    nano 版在 CPU 上才跑得動即時；換 yolov8s/m 精度更好但速度掉一個檔次。
    `--imgsz` 是速度/精度最直接的旋鈕：640 → 960 → 1280 速度約 37 / 19 / 11 FPS。

標註慣例
    # [原文]  簡報截圖上讀得到的程式碼
    # [補完]  截圖被滾動箭頭遮住／講義省略，依框架標準介面補完

簡報對應：第 7 頁（Yolo (Recommended)），四行程式，無遮擋。
"""

from __future__ import annotations

import argparse
import os
import sys


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description="YOLOv8 偵測（單張圖或 webcam）")
    parser.add_argument("--source", default="0",
                        help="圖片/影片路徑，或 webcam index（例如 0）")
    parser.add_argument("--model", default="yolov8n.pt")
    parser.add_argument("--imgsz", type=int, default=640)
    parser.add_argument("--conf", type=float, default=0.25)
    parser.add_argument("--show", action="store_true", help="彈窗顯示（需要 GUI）")
    parser.add_argument("--no-show", action="store_true", help="不彈窗（headless 機器）")
    parser.add_argument("--max-frames", type=int, default=0, help="0=不限（webcam 測試用）")
    parser.add_argument("--out-dir", default="out/yolo")
    args = parser.parse_args(argv)

    from ultralytics import YOLO

    # [原文] 簡報的四行：載入 model、predict(source=0, show=True)
    model = YOLO(args.model)

    source: object = args.source
    if args.source.isdigit():
        source = int(args.source)

    show = args.show and not args.no_show
    os.makedirs(args.out_dir, exist_ok=True)

    print(f"model={args.model} source={source} imgsz={args.imgsz} conf={args.conf}")
    frames = 0

    # [原文] model.predict(source=0, show=True) —— 加上實用參數
    results = model.predict(
        source=source,
        imgsz=args.imgsz,
        conf=args.conf,
        stream=True,   # [補完] 一律 stream：webcam/長影片才不會把結果累積在記憶體
        show=show,
        verbose=False,
    )

    for result in results:
        frames += 1
        names = result.names
        boxes = result.boxes
        n = 0 if boxes is None else len(boxes)
        top = 0.0 if n == 0 else float(boxes.conf.max())

        # [補完] 速度要用 ultralytics 自己的 result.speed，不要用「總時間 / 張數」：
        #        那樣會把模型載入與第一次 warmup 算進去（單張圖會量到 2 FPS 這種假數字）
        infer_ms = float((getattr(result, "speed", {}) or {}).get("inference", 0.0))
        fps = 1000.0 / infer_ms if infer_ms > 0 else 0.0
        print(f"frame {frames:04d}: {n} 個物件, 最高信心 {top:.2f}, "
              f"推論 {infer_ms:.1f} ms ({fps:.1f} FPS)")

        if show:
            continue
        path = os.path.join(args.out_dir, f"pred_{frames - 1:04d}.jpg")
        result.save(filename=path)  # [補完] headless 時存檔，不然什麼都看不到

        if args.max_frames and frames >= args.max_frames:
            break

    if not show:
        print(f"\n已存 {frames} 張到 {args.out_dir}/")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

## 內嵌：`src/06_tf_cnn_mnist.py`

In [ ]:
%%writefile src/06_tf_cnn_mnist.py
#!/usr/bin/env python3
"""06_tf_cnn_mnist.py — TensorFlow/Keras CNN 數字分類（MNIST）。

簡報 §七 的五個階段：載入資料 → 前處理 → 建模型 → 訓練 → 單張推論。

用法:
    python 06_tf_cnn_mnist.py                 # 5 epochs（簡報設定）
    python 06_tf_cnn_mnist.py --epochs 2      # 快速煙霧測試
    python 06_tf_cnn_mnist.py --no-plot       # 不畫圖

Python 版本
    TensorFlow 沒有 3.15 的 wheel（2026-11 實測），所以這支要用 3.11／3.12 跑：
        uv run --no-project --python 3.11 --with tensorflow-cpu -- python 06_tf_cnn_mnist.py

標註慣例
    # [原文]  簡報截圖上讀得到的程式碼
    # [補完]  截圖被滾動箭頭遮住／講義省略，依框架標準介面補完

簡報對應：第 8–17 頁（Tensorflow for CNN）。注意第 8 與第 9 頁是同一張截圖。
"""

from __future__ import annotations

import argparse
import os
import sys


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description="TF/Keras MNIST CNN（簡報 §七）")
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--batch-size", type=int, default=32, help="Keras fit 預設值")
    parser.add_argument("--no-plot", action="store_true", help="不輸出預測圖")
    parser.add_argument("--out-dir", default="out/mnist_tf")
    args = parser.parse_args(argv)

    import numpy as np
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    from tensorflow.keras.datasets import mnist

    print(f"Python {sys.version.split()[0]} / TF {tf.__version__} / Keras {keras.__version__}")

    # [原文] 1. 載入資料
    (x_train, y_train), (x_test, y_test) = mnist.load_data()
    print("Training Data Shape:", x_train.shape)   # (60000, 28, 28)
    print("Test Data Shape:", x_test.shape)        # (10000, 28, 28)

    # [原文] 2. 前處理：多一維是通道數，除以 255 是正規化到 0～1
    x_train = x_train.reshape(-1, 28, 28, 1) / 255.0
    x_test = x_test.reshape(-1, 28, 28, 1) / 255.0

    # [原文] 3. 建立模型
    model = keras.Sequential([
        layers.Conv2D(32, (3, 3), activation="relu", input_shape=(28, 28, 1)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])
    # [補完] 印出參數量：這是「兩套框架的架構真的等價」最硬的證據
    #        （PyTorch 版同樣是 225,034；loss 曲線接近只能算同量級，不能當證明）
    print(f"Param count: {model.count_params():,}")

    # [原文] 4. 編譯與訓練
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    history = model.fit(
        x_train, y_train,
        epochs=args.epochs,
        batch_size=args.batch_size,
        validation_split=0.1,
        verbose=2,
    )

    # [補完] 簡報沒貼評估段落，但它的預期輸出一開始就是 test accuracy
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"\nTest loss: {test_loss:.4f}")
    print(f"Test accuracy: {test_acc:.4f}")

    # [原文] 5. 單張推論
    index = 0
    prediction = model.predict(x_test[index:index + 1], verbose=0)
    print("Predicted Class:", int(np.argmax(prediction)))
    print("Actual Class:", int(y_test[index]))

    if not args.no_plot:
        try:
            import matplotlib
            matplotlib.use("Agg")
            import matplotlib.pyplot as plt

            os.makedirs(args.out_dir, exist_ok=True)
            path = os.path.join(args.out_dir, "prediction.png")
            plt.figure(figsize=(3, 3))
            plt.imshow(x_test[index].reshape(28, 28), cmap="gray")
            plt.title(f"pred={int(np.argmax(prediction))} / actual={int(y_test[index])}")
            plt.axis("off")
            plt.tight_layout()
            plt.savefig(path, dpi=120)
            print(f"已存檔: {path}")
        except Exception as exc:  # matplotlib 不是必要相依
            print(f"[提醒] 沒畫圖（{type(exc).__name__}），不影響結果")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

## 內嵌：`src/07_torch_cnn_mnist.py`

In [ ]:
%%writefile src/07_torch_cnn_mnist.py
#!/usr/bin/env python3
"""07_torch_cnn_mnist.py — PyTorch CNN 數字分類（MNIST）。

同樣五個階段，換 PyTorch 寫：載入資料 → 前處理 → 建模型 → 訓練 → 單張推論。

用法:
    python 07_torch_cnn_mnist.py                # 5 epochs
    python 07_torch_cnn_mnist.py --epochs 1     # 快速煙霧測試
    python 07_torch_cnn_mnist.py --device cuda  # 有 GPU 時

標註慣例
    # [原文]  簡報截圖上讀得到的程式碼
    # [補完]  截圖被滾動箭頭遮住／講義省略，依框架標準介面補完

簡報對應：第 18–25 頁（Or you could use Pytorch）。第 21 頁底部兩行被裁掉、第 19／22 頁被箭頭遮住。
"""

from __future__ import annotations

import argparse
import sys


def build_model():
    """回傳 (model, param_count)。"""
    import torch
    import torch.nn as nn

    # [原文] 建立模型。注意最後一層沒有 softmax：CrossEntropyLoss 內建
    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv2d(1, 32, 3)
            self.pool = nn.MaxPool2d(2, 2)
            self.conv2 = nn.Conv2d(32, 64, 3)
            self.fc1 = nn.Linear(64 * 5 * 5, 128)
            self.fc2 = nn.Linear(128, 10)

        def forward(self, x):
            # 28 → 26 → 13 → 11 → 5，所以特徵圖是 64 × 5 × 5
            x = self.pool(torch.relu(self.conv1(x)))
            x = self.pool(torch.relu(self.conv2(x)))
            x = x.view(-1, 64 * 5 * 5)
            x = torch.relu(self.fc1(x))
            return self.fc2(x)

    model = Net()
    return model, sum(p.numel() for p in model.parameters())


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description="PyTorch MNIST CNN（簡報 §八）")
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--lr", type=float, default=0.001)
    parser.add_argument("--device", default="cpu", choices=["cpu", "cuda"])
    parser.add_argument("--data-dir", default="./data")
    parser.add_argument("--out", default="", help="模型權重輸出檔（預設不存）")
    args = parser.parse_args(argv)

    import torch
    import torch.nn as nn
    import torch.optim as optim
    import torchvision
    import torchvision.transforms as transforms

    print(f"PyTorch {torch.__version__} / torchvision {torchvision.__version__}")
    device = torch.device(args.device)
    if args.device == "cuda" and not torch.cuda.is_available():
        print("[提醒] CUDA 不可用，退回 CPU", file=sys.stderr)
        device = torch.device("cpu")

    # [原文] 載入資料：ToTensor() 同時做 (H,W,C)→(C,H,W) 與 /255
    transform = transforms.ToTensor()
    train_dataset = torchvision.datasets.MNIST(
        root=args.data_dir, train=True, download=True, transform=transform)
    test_dataset = torchvision.datasets.MNIST(
        root=args.data_dir, train=False, download=True, transform=transform)

    print("Training samples:", len(train_dataset))    # 60000
    print("Test samples:", len(test_dataset))         # 10000
    print("Image shape:", train_dataset[0][0].shape)  # torch.Size([1, 28, 28])

    model, params = build_model()
    model.to(device)
    print(model)
    # [補完] 印參數量：與 TF 版同樣是 225,034，這才是架構等價的證據
    print(f"Param count: {params:,}")

    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=args.batch_size, shuffle=True)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=512)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=args.lr)

    # [原文] 訓練迴圈五步：zero_grad → forward → loss → backward → step
    for epoch in range(args.epochs):
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch + 1}/{args.epochs}, Loss: {running_loss / len(train_loader):.4f}")

    # [補完] 簡報的輸出有 test accuracy，但截圖裡沒有評估迴圈 —— 這段就是缺的那塊
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            correct += (torch.argmax(model(images), dim=1) == labels).sum().item()
    total = len(test_dataset)
    print(f"Test accuracy: {correct / total:.4f} ({correct}/{total})")

    # [原文] 單張推論：unsqueeze(0) 加上 batch 維度
    image, actual_label = test_dataset[0]
    with torch.no_grad():
        output = model(image.unsqueeze(0).to(device))
        predicted_label = torch.argmax(output, dim=1).item()
    print("Predicted Class:", predicted_label)
    print("Actual Class:", actual_label)

    if args.out:
        torch.save(model.state_dict(), args.out)
        print(f"已存權重: {args.out}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

## 內嵌：`bench/bench_detection.py`

In [ ]:
%%writefile bench/bench_detection.py
#!/usr/bin/env python3
"""bench/bench_detection.py — 同一段畫面的偵測模型基準線。

為什麼要有這支：把「FPS」放在一起比很危險，除非<strong>同一張影格、同一個門檻、同一台機器</strong>。
這支就是筆記 §六 那張表的來源——它對每張圖跑同樣的模型、量純推論時間、
順便印出「最高信心」與「≥門檻有幾個」，所以「快但什麼都沒看到」會直接現形。

用法:
    python bench/bench_detection.py --frame frame.jpg
    python bench/bench_detection.py --camera 0 --repeat 5      # 現場抓一張再跑
    python bench/bench_detection.py --frame frame.jpg --yolo    # 加測 YOLOv8n（需 ultralytics）
    python bench/bench_detection.py --frame frame.jpg --json out.json

輸出：一張 Markdown 表 + （可選）JSON。時間是「模型 forward 的平均值」，
不含攝影機讀取與畫框，所以會比 04_torch_webcam_detection.py 顯示的 FPS 高。
"""

from __future__ import annotations

import argparse
import json
import os
import sys
import time

CASES = [
    ("fasterrcnn_resnet50_fpn", {}),
    ("fasterrcnn_resnet50_fpn", {"min_size": 400, "max_size": 667}),
    ("fasterrcnn_mobilenet_v3_large_fpn", {}),
    ("fasterrcnn_mobilenet_v3_large_320_fpn", {}),
    ("ssdlite320_mobilenet_v3_large", {}),
]


def grab_frame(camera: int) -> str:
    """抓一張實機影格存檔，回傳路徑（沿用 04 的 pick_camera）。"""
    import cv2

    sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))) + "/src")
    from importlib import import_module

    pick = import_module("04_torch_webcam_detection").pick_camera
    cap = pick(camera)
    if cap is None:
        print("[錯誤] 找不到有畫面的攝影機（先跑 src/00_check_env.py）", file=sys.stderr)
        raise SystemExit(2)
    try:
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
        for _ in range(10):
            cap.read()
        ok, frame = cap.read()
    finally:
        cap.release()
    if not ok or frame is None:
        print("[錯誤] 讀不到 frame", file=sys.stderr)
        raise SystemExit(2)
    path = os.path.abspath("bench_frame.jpg")
    cv2.imwrite(path, frame)
    print(f"captured {path} ({frame.shape[1]}x{frame.shape[0]}, brightness {frame.mean():.0f})")
    return path


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description="同一段畫面的偵測模型基準線")
    parser.add_argument("--frame", help="影格路徑；沒給就用 --camera 現場抓")
    parser.add_argument("--camera", type=int, default=0)
    parser.add_argument("--repeat", type=int, default=5, help="每個模型計時幾次（預設 5）")
    parser.add_argument("--warmup", type=int, default=2)
    parser.add_argument("--threshold", type=float, default=0.5)
    parser.add_argument("--yolo", action="store_true", help="加測 YOLOv8n 640/960/1280")
    parser.add_argument("--json", dest="json_out", default="")
    args = parser.parse_args(argv)

    import cv2
    import torch
    from torchvision.models import detection
    from importlib import import_module

    sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))) + "/src")
    # 類別名稱沿用 03 的邏輯：SSD 家族沒有 .weights，要從 WeightsEnum 拿
    cats_helper = import_module("03_torch_object_detection_image")

    path = args.frame or grab_frame(args.camera)
    bgr = cv2.imread(path)
    if bgr is None:
        print(f"[錯誤] 讀不到圖片: {path}", file=sys.stderr)
        return 2
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    tensor = torch.from_numpy(rgb).permute(2, 0, 1).float().div(255.0).unsqueeze(0)
    print(f"frame: {rgb.shape[1]}x{rgb.shape[0]}, threshold={args.threshold}, "
          f"repeat={args.repeat}\n")

    rows = []
    for fn, kwargs in CASES:
        label = fn + (" + min_size=400" if kwargs.get("min_size") == 400 else "")
        try:
            model = getattr(detection, fn)(weights="DEFAULT", **kwargs)
            model.eval()
            with torch.no_grad():
                for _ in range(args.warmup):
                    model(tensor)
                t0 = time.time()
                for _ in range(args.repeat):
                    pred = model(tensor)[0]
                dt = (time.time() - t0) / args.repeat
        except Exception as exc:
            print(f"| {label} | — | — | — | — | FAILED {type(exc).__name__} |")
            continue

        scores = pred["scores"].numpy()
        labels = pred["labels"].numpy()
        cats = cats_helper.categories_for(fn, model)
        keep = scores >= args.threshold
        names = sorted({cats[int(l)] for l in labels[keep]}) if cats else []
        rows.append({"model": label, "ms": round(dt * 1000, 1), "fps": round(1 / dt, 1),
                     "max_score": round(float(scores.max()), 3),
                     "n": int(keep.sum()), "labels": names})
        print(f"| `{label}` | {dt*1000:.0f} ms | {1/dt:.1f} FPS | "
              f"{scores.max():.3f} | {int(keep.sum())} | "
              f"{'、'.join('`' + n + '`' for n in names) or '—'} |")
        del model

    if args.yolo:
        from ultralytics import YOLO

        model = YOLO("yolov8n.pt")
        for imgsz in (640, 960, 1280):
            r = model.predict(source=rgb, imgsz=imgsz, conf=0.25, verbose=False)[0]
            ms = float(r.speed["inference"])
            confs = r.boxes.conf.cpu().numpy() if r.boxes is not None and len(r.boxes) else []
            names = sorted({r.names[int(c)] for c in r.boxes.cls}) if len(confs) else []
            keep = int((confs >= args.threshold).sum()) if len(confs) else 0
            top = float(max(confs)) if len(confs) else 0.0
            rows.append({"model": f"yolov8n imgsz={imgsz}", "ms": round(ms, 1),
                         "fps": round(1000 / ms, 1), "max_score": round(top, 3),
                         "n": keep, "labels": names})
            print(f"| `yolov8n imgsz={imgsz}` | {ms:.0f} ms | {1000/ms:.1f} FPS | {top:.3f} | "
                  f"{keep} | {'、'.join('`' + n + '`' for n in names) or '—'} |")

    if args.json_out:
        json.dump(rows, open(args.json_out, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
        print(f"\n已存 {args.json_out}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

## 4. 自檢（用內嵌的 `00_check_env.py`）

Colab 沒有攝影機，所以加 `--skip-camera`。

In [ ]:
%run src/00_check_env.py --skip-camera

## 5. 物件偵測 · 單張圖（torchvision，簡報第 5 頁）

In [ ]:
%run src/03_torch_object_detection_image.py --image bus.jpg --model ssdlite320 --threshold 0.4

## 6. 物件偵測 · YOLOv8（簡報第 7 頁）

In [ ]:
%run src/05_yolo_detection.py --source bus.jpg --imgsz 640

## 7. 物件偵測 · TensorFlow / TF-Hub（簡報第 2 頁）

In [ ]:
%run src/01_tf_object_detection_image.py --image bus.jpg --threshold 0.5

## 8. 「即時偵測」（簡報第 3／6 頁）→ 用影片取代攝影機

題目若要求 webcam 即時偵測：Colab 沒有相機，把來源換成 `clip.mp4`，流程一行都不用改。

In [ ]:
%run src/04_torch_webcam_detection.py --video clip.mp4 --model ssdlite320

## 9. CNN 分類（簡報第 8–25 頁）· 考試時間短，1 epoch 就好

In [ ]:
%run src/06_tf_cnn_mnist.py --epochs 1 --no-plot

In [ ]:
%run src/07_torch_cnn_mnist.py --epochs 1

## 10. 附加：同一段畫面的模型基準線（可選，會下載 4 組權重）

In [ ]:
%run bench/bench_detection.py --frame bus.jpg --repeat 3

## 11. 看結果圖（Colab 沒有視窗，用 display）

In [ ]:
from IPython.display import Image, display
import glob, os
for p in ["out/torch_detection.jpg", "out/tf_detection.jpg"] + sorted(glob.glob("out/yolo/*.jpg"))[:1]:
    if os.path.exists(p):
        print(p)
        display(Image(p, width=640))

## 考試當天速查

| 症狀 | 解法 |
|---|---|
| `cv2.error ... not implemented ... GTK+` | 不要 `imshow`；用 `--no-show` 或預設（本 notebook 都已避開），圖用 `display(Image(...))` |
| `[錯誤] 找不到有畫面的攝影機` | Colab 沒 webcam → `--video clip.mp4`（第 8 格） |
| `No module named 'ultralytics'` | 第 2 格 `%pip install -q ultralytics` |
| `No module named 'cv2'` 而且你用了 `!python` | 改成 `%run`（`!python` 可能不是 kernel 那顆） |
| TF-Hub 卡住 | 首次要下載 ~20 MB，等它；或先跑 torchvision 那條線 |
| TF 印 `CUDA Runtime error ... GPU will not be used` | 正常（CPU runtime 的 stub），不影響結果；要 GPU 就換執行環境 |
| MNIST 下載失敗 | 重跑該格 |
| Runtime disconnected | `Runtime → Run all`（全部重跑 < 2 分鐘） |
| 想更快 | `執行環境 → 變更執行環境 → T4 GPU`（本 notebook 的模型在 CPU 上也都跑得動） |

**貼答案常用的輸出**：`03`→`person 0.989 / bus 0.985`；`05`→`6 個物件、最高 0.87`；
`06`／`07`→`Test accuracy`（1 epoch 約 0.98）。CPU 上分別約 30 ms / 28 ms / 20 秒。